# 노래 구간 비교 실험: 보컬 분리 전/후 인식 (이슈 #18)

곡의 한 구간을 잘라 **보컬만 분리**하고(Demucs), 원곡과 분리한 보컬을 **같은 조건으로 인식해 정답과 비교**합니다. 터미널 명령(`scripts/separate_clip.py`, `scripts/compare_clips.py`)과 같은 모듈(`separate.py`, `clip_compare.py`)을 쓰며, 이 노트북은 **듣기·표 확인·그림**을 한 화면에서 하려는 것입니다. 절차와 기록 양식은 `docs/노래_구간_비교_실험_절차.md`, 판단은 검토 문서 13.5입니다.

**순서**: ① 설정 → ② 구간 정하기(`CLIPS`) → ③ 분리 실행 → ④ 들어 보기 → ⑤ 정답 파일 준비(편집기로 직접 받아 적기) → ⑥ 비교 실행 → ⑦ 스펙트로그램.

**주의 (공개 저장소)**
- 곡·구간·정답·인식 문장은 모두 git 제외 폴더(`data/`, `results/`)에만 저장됩니다. 이 노트북의 **출력에는 오디오 플레이어(곡 소리가 base64로 저장됨)가 들어갈 수 있으므로** 커밋하기 전에 출력을 지우고 `scripts/scan_outputs.py`로 확인합니다(정리본만 커밋).
- 가사(정답, 인식 문장)는 이 노트북 화면에 찍지 않도록 만들어져 있습니다. 점수 숫자만 나옵니다.
- 분리와 인식은 모두 내 컴퓨터에서만 돌고 외부 API로 나가지 않습니다(과금 없음).
- `.py`를 고친 뒤에는 자동으로 다시 읽도록 `autoreload`를 켜 두었지만, 이상하면 커널을 재시작합니다.

## 이 노트북은 무엇을 하는가 (한눈에 보기, 2026-11-03 저녁)

**질문 하나**: 노래에서 반주를 떼어 낸 목소리(분리 보컬)가 원곡 그대로보다 Whisper(STT)에게 **가사를 더 잘 받아 적게 하는가?** 그렇다면 그 문장으로 연습하는 흐름(이슈 #31·#32)으로 이어 가고, 아니라면 분리 없이 가는 근거가 됩니다. 이 노트북은 그 질문을 **내 곡 구간 8개의 숫자로 확인**하는 곳입니다(이슈 #18).

### 전체 순서

| 단계 | 하는 일 | 걸리는 시간 |
|---|---|---|
| ② `CLIPS` | 곡 번호·시작·길이로 구간을 정함 | 직접 |
| ③a 자르기 | 구간을 잘라 `cNN_mix.wav` 저장 | 몇 초 |
| ④ 듣기 | 원곡 구간이 가사 부분인지 귀로 확인 | 구간당 1분 |
| ③b 분리 | Demucs로 보컬만 분리해 `cNN_vocals.wav` 저장 (별도 프로세스) | 구간당 1~2분 |
| ⑤ 정답 | 원곡을 듣고 가사를 받아 적음 (`truth/cNN.txt`, 화면에는 단어 수만) | 구간당 5~10분 |
| **⑥ 비교** | **원곡 vs 보컬을 같은 조건으로 인식해 정답과 비교** | 조건에 따라 5~40분 |
| ⑦ 그림 | 원곡과 보컬의 스펙트로그램 | 몇 초 |

### ⑥ 비교 실행은 정확히 무엇을 비교하는가

비교의 **핵심은 소리**입니다. 나머지는 그 결론이 조건에 따라 달라지는지 보려고 바꿔 보는 값입니다.

| 바꾸는 값 | 값 | 보려는 것 |
|---|---|---|
| **소리** | 원곡 구간(`mix`) vs 분리한 보컬(`vocals`) | **분리가 도움이 되는가** (핵심) |
| 모델 크기 | `tiny`, `base`, `small` | 큰 모델은 반주에 더 강해서 분리의 이득이 줄어드는가 [추정] |
| VAD | 끔 / 켬 | 말소리 없는 구간을 거르는 처리가 노래에서 도움이 되는가 |
| `temperature` (추가 예정) | 기본(폴백) / 0.0 고정 | 결과가 실행마다 흔들리는 정도 (아래 "지금까지 결과") |

- **정답**: 내가 **원곡을 들으며** 받아 적은 가사입니다(분리 결과에 정답이 끌려가지 않게).
- **점수**: 인식된 문장과 정답을 단어 단위로 비교한 오류율 **WER**(치환·삭제·삽입 ÷ 정답 단어 수)와 글자 단위 **CER**. 낮을수록 정답에 가깝습니다. 1을 넘을 수 있습니다. 전체는 **micro**(오류 합 ÷ 정답 단어 합)를 기본으로, 구간별 평균 **macro**를 보조로 봅니다.
- 한 구간 × 한 조건 = 인식 한 번 = 결과 한 줄입니다(`results/song_compare/metrics.csv`). 인식된 문장(가사)은 `texts.csv`에만 저장되고 화면에는 나오지 않습니다.
- **점수는 발음이나 노래 실력이 아니라, STT가 정답 가사를 얼마나 되살렸는가**입니다.

### 왜 하는가
1. **가설을 검증**하려고 합니다. "분리하면 좋아진다"는 직관이고, 분리가 만든 어색한 소리가 오히려 인식을 해칠 수도 있습니다(검토 문서 15.4).
2. **대조군**이 있습니다. c07(아카펠라)은 반주가 없어서 분리가 필요 없는 경우입니다. 여기서도 분리가 크게 달라지면 분리 자체의 영향을 의심합니다.
3. 결정이 걸려 있습니다. 분리가 쓸 만하면 #31·#32로 가고, 아니면 계획서의 축소 규칙대로 분리 없이 가거나 구간을 줄입니다.

### 지금까지 결과 (참고 baseline, tiny, VAD 끔, 8구간·정답 231단어)

| 실행 | 원곡 오류 / micro WER | 보컬 오류 / micro WER |
|---|---|---|
| 터미널(`compare_clips.py`) | 117 / 0.506 | 140 / 0.606 |
| 이 노트북 ⑥ | 126 / 0.545 | 160 / 0.693 |

- **같은 입력인데 두 실행의 값이 달랐습니다.** 원인을 확인했습니다 [확인]: faster-whisper가 기본으로 쓰는 **온도 폴백**(어려운 소리에서 무작위 샘플링으로 재시도)이 노래에서 발동합니다. 같은 16개 파일을 두 번 인식하면 **4개의 문장이 달랐고, `temperature=0.0`이면 16개 모두 같았습니다.** 연습 문장 30건에서는 이런 일이 없었습니다(말소리라서 폴백이 거의 안 걸림).
- 따라서 **실행 간 차이(0.04~0.09)가 원곡-보컬 차이(0.10~0.15)와 비슷한 크기**여서, 지금 값만으로 "분리하면 나빠진다"고 말할 수 없습니다. 결론은 보류입니다.

### 커밋 직전에 하던 일 (정리)
1. 자르기(`cut_clip`)와 분리(`separate_clip`)를 나눔: c05·c06이 가사 구간이 아니어서 분리 시간이 낭비된 것을 계기로, 먼저 듣고 맞는 구간만 분리하도록 바꿈.
2. ③b를 별도 프로세스로 바꿈: 분리(torch)와 인식(Whisper)을 한 커널에서 쓰면 OpenMP 충돌로 커널이 죽음(이슈노트 #15).
3. 위 결과 불일치의 원인(온도 폴백)을 찾아 문서를 정정함(검토 문서 13.5).
4. **C안 결정**: `temperature=0.0` 고정을 주 결과로 하고 기본 설정 3회 범위를 민감도 확인으로 병행합니다. 구현은 커밋 정리 뒤에 합니다.
5. 변경 사항을 주제별로 커밋 정리.

**다음 할 일**은 `docs/작업_일지.md` 3장의 11-03 항목에 있습니다.

## ① 설정

In [ ]:
# ① 설정: 프로젝트 루트로 이동하고 모듈을 불러옵니다. (이 노트북은 experiments/ 안에 있어서 상대경로 data/...가 맞도록 루트로 옮깁니다.)
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "separate.py").exists() and root != root.parent:
    root = root.parent                      # 위 폴더로 올라가며 프로젝트 루트(separate.py가 있는 곳)를 찾는다
os.chdir(root)
sys.path.insert(0, str(root))

try:                                        # .py를 고치면 커널을 재시작하지 않아도 다시 읽게 한다(노트북에서만 동작)
    ip = get_ipython()
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")
except Exception:
    pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

import audio            # 파일 읽기·16 kHz 모노 변환(수업 코드)
import clip_compare     # 구간 × 소리 × 모델 크기 × VAD 비교
import separate         # 곡 구간 자르기 + 보컬 분리(Demucs)
import visualisation    # 스펙트로그램·Log-Mel 그림

songs = separate.list_songs("en")           # data/songs/en 의 곡 파일(이름순). 번호는 이 목록의 순서(1부터)
print("프로젝트 루트:", Path.cwd().name, "| 영어 곡", len(songs), "개 | 작업 폴더", separate.WORK_DIR)


## 곡 목록 (번호 = `CLIPS`의 `song` 값)

아래 표가 영어 곡 전체 목록입니다. **번호**를 `CLIPS`의 `song`에 적고, **길이(초)**를 보고 `start`가 곡 길이를 넘지 않게 합니다(간주가 긴 곡은 들어 보고 정합니다). 이 표의 출력에는 곡 파일 이름이 나오므로 커밋할 때는 출력을 지운 정리본만 올립니다. 곡을 폴더에 더하거나 빼면 번호가 바뀌니, 구간을 분리한 뒤에는 `data/song_work/clips.csv`의 파일 이름을 기준으로 봅니다.

In [ ]:
# 곡 목록 표: 번호, 파일 이름, 형식, 길이(초), 크기(MB). 길이는 파일 머리글에서 읽어서 빠르다(전체 디코딩 안 함).
import av   # PyAV: faster-whisper가 함께 깐 디코더 패키지. 여기서는 길이만 읽는다.

def song_seconds(path):
    try:
        with av.open(str(path)) as ct:
            return round(ct.duration / 1e6, 1) if ct.duration else None   # duration은 마이크로초 단위
    except Exception:
        return None

song_table = pd.DataFrame({
    "번호": range(1, len(songs) + 1),
    "파일 이름": [p.name for p in songs],
    "형식": [p.suffix.lower().lstrip(".") for p in songs],
    "길이(초)": [song_seconds(p) for p in songs],
    "크기(MB)": [round(p.stat().st_size / 1e6, 1) for p in songs],
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 200):
    display(song_table.set_index("번호"))

- 유형별 곡 번호 목록
  - 가사가 또렷한 곡: 12(gospel), 58(bossa nova/folk), 48(rock), 9(ballad), 16(folk)
  - 반주가 두꺼운 곡: 1(brass), 18(synth), 26(live), 15(jazz), 23(hip hop), 53(synth)
  - 합창/코러스/겹치는 목소리: 35(musical/chorus), 51(accapella), 52(duet), 49(chorus), 44(duet)
  - 랩: 55(rap), 3(말+노래), 13(rap)

## ② 구간 정하기

`CLIPS`에 구간을 한 줄씩 적습니다. `song`은 ①의 목록 번호, `start`는 구간 시작(초), `duration`은 길이(초, 10~20 권장), `id`는 구간 이름(`c01`, `c02`, …)입니다. 번호는 폴더의 곡이 바뀌면 달라지므로, 분리하면 `data/song_work/clips.csv`에 **파일 이름**이 함께 기록됩니다.

구간 선택 기준은 `docs/노래_구간_비교_실험_절차.md` 2절입니다(가사가 또렷한 곡, 반주가 두꺼운 곡, 합창이 있는 구간을 섞기).

In [ ]:
# ② 구간 목록. 아래 예시는 주석입니다. 지우고 내 구간으로 채우세요.
CLIPS = [
    # {"id": "c01", "song": 6, "start": 60, "duration": 20},
    # {"id": "c02", "song": 12, "start": 45, "duration": 15},
    {"id": "c01", "song": 9,  "start": 61,  "duration": 20},   # 또렷
    {"id": "c02", "song": 58, "start": 96,  "duration": 20},   # 또렷
    {"id": "c03", "song": 48, "start": 102, "duration": 20},   # 또렷
    {"id": "c04", "song": 26, "start": 60, "duration": 20},   # 반주 두꺼움
    {"id": "c05", "song": 18, "start": 20,  "duration": 20},   # 반주 두꺼움
    {"id": "c06", "song": 1,  "start": 31,  "duration": 20},   # 반주 두꺼움
    {"id": "c07", "song": 51, "start": 71,  "duration": 20},   # 합창 (아카펠라, 대조군)
    {"id": "c08", "song": 35, "start": 60,  "duration": 20},   # 합창
]
OVERWRITE = False       # True면 이미 분리한 구간도 다시 분리합니다(start를 고쳤을 때)

## ③ (이전 방식) 자르기 + 분리를 한 번에 — c01~c08 첫 실행 기록

아래 셀은 **자르기와 분리를 한 번에** 하던 첫 방식입니다. 출력에 분리 시간 기록이 남아 있어 그대로 둡니다. **다시 실행하지 않습니다.** 앞으로는 아래 ③a(자르기) → ④(듣기) → ③b(분리) 순서로 합니다. 구간을 듣고 확인한 뒤에 분리하면 어긋난 구간의 분리 시간(구간당 1분 안팎)을 아낄 수 있습니다.

In [ ]:
# # ③ 구간마다 분리: separate.process_clip → data/song_work/<id>_mix.wav, <id>_vocals.wav, clips.csv
# if not CLIPS:
#     print("CLIPS가 비어 있습니다. ②에서 구간을 적고 다시 실행하세요.")
# for c in CLIPS:
#     mix = Path(separate.WORK_DIR) / f"{c['id']}_mix.wav"
#     if mix.exists() and not OVERWRITE:
#         print(f"{c['id']}: 이미 있음 → 건너뜀 (다시 하려면 OVERWRITE = True)")
#         continue
#     r = separate.process_clip(songs[c["song"] - 1], c["start"], c["duration"], c["id"], overwrite=OVERWRITE)
#     print(f"{c['id']}: 분리 {r['seconds']:.1f}초 ({r['seconds'] / c['duration']:.2f}배)")

## ③a 구간 자르기 (몇 초)

`CLIPS`의 구간을 **자르기만** 합니다(분리 안 함). 곡 번호·`start`·`duration`이 이미 잘라 둔 것과 같은 구간은 건너뛰고, **바뀐 구간만 다시 자릅니다**(그 구간의 옛 보컬 파일은 지웁니다). 그래서 `start`를 고친 뒤 이 셀만 다시 실행하면 되고 `OVERWRITE`는 켜지 않아도 됩니다. 일부 구간만 하려면 `ONLY = ["c05", "c06"]`처럼 적습니다.
다 자른 뒤에는 ④에서 `<id>_mix.wav`를 들어 **가사 부분이 맞는지 확인**합니다.

In [ ]:
# ③a 자르기만: separate.cut_clip → data/song_work/<id>_mix.wav, clips.csv. 같은 구간은 건너뛰고 바뀐 구간만 다시 자른다.
ONLY = None      # None이면 CLIPS 전부. 일부만: 예) ["c05", "c06"]
targets = [c for c in CLIPS if ONLY is None or c["id"] in ONLY]
if not targets:
    print("대상 구간이 없습니다. ②의 CLIPS와 ONLY를 확인하세요.")
for c in targets:
    song = songs[c["song"] - 1]
    if separate.is_cut_current(song, c["start"], c["duration"], c["id"]):
        print(f"{c['id']}: 이미 같은 구간을 잘라 둠 → 건너뜀")
        continue
    r = separate.cut_clip(song, c["start"], c["duration"], c["id"], overwrite=True)   # 바뀐 구간이면 다시 자르고 옛 보컬은 지움
    print(f"{c['id']}: 곡 {c['song']}번 {c['start']}초부터 {r['seconds']:.0f}초 잘랐습니다 → ④에서 {c['id']}_mix.wav를 들어 보세요")

## ③b 분리 (구간당 1~2분)

④에서 **들어 보고 가사 부분이 맞다고 확인한 구간만** 분리합니다. 이미 보컬 파일이 있는 구간은 건너뜁니다. 어긋난 구간은 `start`를 고쳐 ③a를 다시 실행한 뒤(그 구간만 다시 잘림) 이 셀을 실행합니다. 일부만 분리하려면 ③a의 `ONLY`를 바꿉니다. 셀이 끝날 때까지 기다립니다.

**분리는 별도 프로세스(`scripts/separate_clip.py --separate-only`)로 실행합니다.** 분리에 쓰는 `torch`가 이 노트북 커널에 올라가면, 같은 커널에서 ⑥(Whisper 인식)을 실행할 때 OpenMP 충돌(`OMP: Error #15`)로 커널이 죽기 때문입니다(`docs/개발환경_이슈노트.md` #15). 이렇게 하면 ③b와 ⑥을 어떤 순서로 실행해도 됩니다.

In [ ]:
# ③b 분리: 구간마다 터미널 스크립트(scripts/separate_clip.py --separate-only)를 별도 프로세스로 실행한다.
# 왜 subprocess인가: torch(Demucs)를 이 커널에 불러오면 ⑥의 Whisper(CTranslate2)와 OpenMP가 충돌해 커널이 죽는다(이슈노트 #15).
# 별도 프로세스는 끝나면 사라지므로 이 커널에는 torch가 올라가지 않는다.
import subprocess

targets = [c for c in CLIPS if ONLY is None or c["id"] in ONLY]
for c in targets:
    mix_p, voc_p = separate.clip_paths(c["id"])
    if not mix_p.exists():
        print(f"{c['id']}: 아직 자르지 않았습니다 → ③a를 먼저 실행하세요")
        continue
    if voc_p.exists():
        print(f"{c['id']}: 이미 분리함 → 건너뜀")
        continue
    print(f"{c['id']}: 분리 중(1~2분)…")
    r = subprocess.run([sys.executable, "scripts/separate_clip.py", "--id", c["id"], "--separate-only"],
                       capture_output=True, text=True)       # sys.executable = 이 커널의 파이썬(voice-ai). 출력은 아래에서 골라 보여 준다
    if r.returncode != 0:
        print(f"{c['id']}: 실패(종료 코드 {r.returncode})")
        print("\n".join(r.stderr.strip().splitlines()[-5:]))   # 오류 마지막 5줄
        continue
    print("   " + " ".join(l for l in r.stdout.splitlines() if l.startswith("분리")))

## ④ 들어 보기

`LISTEN_ID`를 바꿔 구간을 듣습니다. **③a 직후에는 원곡(mix)만** 있으니 먼저 mix로 구간이 가사 부분인지 확인하고, 맞으면 ③b로 분리한 뒤 보컬(vocals)도 들어 분리 품질(좋음/보통/나쁨)을 봅니다. 어긋나면 `start`를 고치고 ③a를 다시 실행합니다. (이 셀의 출력에는 곡 소리가 들어가므로 커밋 전에 출력을 지웁니다.)

In [ ]:
# ④ 듣기: 구간 원본(mix)과 분리한 보컬(vocals)
LISTEN_ID = "c03"
for name in ("mix", "vocals"):
    p = Path(separate.WORK_DIR) / f"{LISTEN_ID}_{name}.wav"
    if p.exists():
        print(name)
        display(Audio(str(p)))
    else:
        print(f"{p.name} 이 없습니다. ③a(mix) 또는 ③b(vocals)를 먼저 실행하세요.")

## ⑤ 정답 가사 준비 (직접 받아 적기)

아래 셀은 정답 파일 `data/song_work/truth/<id>.txt`의 **빈 파일을 만들고 현재 상태(단어 수)만** 보여 줍니다. 파일은 **VS Code나 텍스트 편집기에서 직접 열어** 구간에서 불린 가사의 단어를 받아 적습니다. **정답은 분리 전 원곡(mix)을 들으며 적습니다**(분리 결과에 정답이 끌려가지 않게 하려는 것)(규칙은 절차 문서 4절: 추임새 제외, 반복은 횟수대로). 가사는 이 화면에 나오지 않습니다. 다 적고 이 셀을 다시 실행해 단어 수와 "준비됨"을 확인합니다.

In [ ]:
# ⑤ 정답 파일 만들기와 상태 표(가사는 찍지 않고 단어 수만)
ids = [c["id"] for c in CLIPS]
truth_dir = Path(separate.WORK_DIR) / "truth"
truth_dir.mkdir(parents=True, exist_ok=True)
rows = []
for cid in ids:
    t = clip_compare.truth_path(cid)
    if not t.exists():
        t.touch()                                           # 빈 파일을 만들어 둔다(편집기로 열 수 있게)
    n_words = len(clip_compare.load_truth(cid).split())
    rows.append({"구간": cid, "정답 파일": str(t), "정답 단어 수": n_words})
ready, missing = clip_compare.available_clips()
for r in rows:
    r["준비됨"] = r["구간"] in ready
    r["빠진 것"] = "/".join(missing.get(r["구간"], [])) or "-"
with pd.option_context("display.max_colwidth", None, "display.width", 200):
    display(pd.DataFrame(rows))

## ⑥ 비교 실행

준비된 구간을 **원곡 / 보컬 × 모델 크기 × VAD**로 인식하고 정답과 비교합니다. 로컬이라 과금은 없습니다. 구간 5개 × 2 × 크기 3 × VAD 2 = 60번 인식에 10~20분 [추정]이 걸릴 수 있으니, 처음에는 `SIZES = ["tiny"]`, `VADS = (False,)`로 짧게 확인합니다. `small`은 처음 쓸 때 내려받습니다.

화면 표의 읽는 법은 절차 문서 6절입니다. 인식된 문장(가사)은 `results/song_compare/texts.csv`에만 저장되고 화면에는 나오지 않습니다.

무엇을 어떻게 비교하는지(소리 × 모델 크기 × VAD, 정답, 점수)는 맨 위 「이 노트북은 무엇을 하는가」에 정리해 두었습니다. 같은 입력이어도 결과가 실행마다 달라질 수 있으므로(온도 폴백) `TEMPS`로 설정을 고르고 **보고할 때 그 설정을 함께 적습니다**.

- **`TEMPS = (0.0,)`(주 결과)**: 폴백 없이 인식해서 같은 조건이면 결과가 항상 같습니다. 모델 크기·VAD·원곡/보컬 비교는 이 설정으로 합니다.
- **`TEMPS = (None,)` + `REPEATS = 3`(확인용)**: faster-whisper 기본 설정을 3번 되풀이해 `micro_min`~`micro_max`로 **흔들림 크기**를 봅니다. 원곡-보컬 차이가 이 범위보다 크면 의미 있는 차이로 봅니다.
- `(0.0, None)`처럼 둘을 한 번에 넣을 수도 있습니다. 인식 횟수가 그만큼 늘어납니다.

In [ ]:
# ⑥ 비교 실행: clip_compare.compare → 점수표(metrics, 가사 없음)와 인식 문장(texts, 가사)
SIZES = ["tiny", "base"]          # 예: ["tiny", "base", "small"] / ["tiny"]  
VADS = (False, True)             # 예: (False, True)  VAD = 말소리 없는 구간을 걸러내는 처리
TEMPS = (None,)              # temperature: 0.0 = 폴백 없이 재현 가능(주 결과) / None = faster-whisper 기본(온도 폴백, 실행마다 달라질 수 있음). 예: (0.0, None)
REPEATS = 3                 # 같은 조건 전체를 되풀이할 횟수. None(기본 폴백)의 흔들림 범위를 보려면 3 이상 (예: TEMPS = (None,), REPEATS = 3)

ready, missing = clip_compare.available_clips()
targets = [i for i in ready if not CLIPS or i in {c["id"] for c in CLIPS}]
if missing:
    print("준비가 덜 된 구간(건너뜀):", {k: "/".join(v) for k, v in missing.items()})
if not targets:
    print("비교할 구간이 없습니다. ③ 분리와 ⑤ 정답을 먼저 준비하세요.")
else:
    metrics, texts = clip_compare.compare(targets, SIZES, VADS, temperatures=TEMPS, repeats=REPEATS)
    out = Path("results/song_compare")
    out.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(out / "metrics.csv", index=False)
    texts.to_csv(out / "texts.csv", index=False)       # 가사: 화면에 찍지 않고 파일로만 저장
    print(f"구간 {len(targets)}개 × 소리 2 × 크기 {len(SIZES)} × VAD {len(VADS)} × temperature {len(TEMPS)} × 반복 {REPEATS} = {len(metrics)}번 인식 완료")
    print("[1] 조건별 요약 (micro = 오류 합 ÷ 정답 단어 합, macro = 구간별 평균, micro_min~max = 반복 간 범위; temp 기본 = 온도 폴백)")
    display(clip_compare.summarize(metrics).round(3))
    print("[2] 구간 × 조건 WER")
    display(clip_compare.pivot_wer(metrics).round(3))

#### 실험1) 결과 검토 (`temperature=0.0`, VAD 끔/켬, tiny·base·small, 구간 8개·정답 231단어, 2026-11-03 밤)

> ※ 실험1은 폴백을 끈(`temperature=0.0`) 진단용 설정입니다. **일부 결론이 실험2(기본 설정)에서 바뀌었으니 바로 아래 「실험2) 결과 검토」의 표를 먼저 보세요.**

**한 줄 결론**
- **VAD를 켜면 원곡은 가사를 통째로 버리고, 분리한 보컬은 환각이 크게 줄어듭니다.** 가장 좋은 조합은 **base + VAD 켬 + 보컬(micro WER 0.494)** 이고, 그래도 단어의 절반은 틀립니다.
- WER이 1을 넘는 큰 값은 **같은 구절을 계속 반복하는 환각 루프** 때문입니다(삽입 40개 이상인 7건 전부).
- "분리하면 STT가 좋아진다"는 가정은 **분리가 STT를 직접 좋게 한다기보다, 분리가 VAD를 쓸 수 있게 해 주고 VAD가 환각을 줄인다**로 읽는 편이 데이터와 맞습니다 **[추정]**.

**조건별 micro WER** (오류 합 ÷ 정답 단어 합, 낮을수록 좋음)

| 모델 | 소리 | VAD 끔 | VAD 켬 | 켰을 때 |
|---|---|---|---|---|
| tiny | 원곡 | 1.026 | 0.788 | 좋아짐 |
| tiny | 보컬 | 0.983 | **0.619** | 좋아짐 |
| base | 원곡 | 0.584 | 0.740 | **나빠짐** |
| base | 보컬 | 1.221 | **0.494** | **크게 좋아짐** |
| small | 원곡 | 0.909 | 1.307 | 나빠짐 |
| small | 보컬 | 0.952 | 1.134 | 나빠짐 |

**관찰 1. VAD 켬은 원곡의 가사를 통째로 지웁니다 [확인].**
- VAD 켬 + 원곡에서 **8구간 중 5구간(c01·c03·c04·c05·c08)이 인식 결과가 비었습니다**(세 모델 모두). 그래서 WER이 정확히 1.00이고 오류의 대부분이 삭제(D)입니다(tiny 원곡: D 31 → 167, 삽입은 147 → 0).
- 같은 구간의 **보컬은 빈 결과가 0건**입니다. 아카펠라 대조군(c07)은 반주가 없어서 원곡도 VAD를 통과합니다(VAD 끔/켬 값이 같음).
- 이유 **[추정]**: Whisper에 들어 있는 VAD는 말소리용이라 반주가 섞인 노래를 말소리로 보지 않습니다. 앞서 6곡으로 한 시험과 같은 방향입니다. 즉 **원곡에는 VAD를 쓸 수 없고, 분리하면 쓸 수 있습니다.**

**관찰 2. 큰 WER은 반복 루프입니다 [확인: 지표].**
- 삽입이 40개 이상인 7건은 모두 **고유 단어 비율이 0.01~0.04, 4단어 연속 반복 비율이 0.98~1.00**입니다. 정답에 없는 말이 새로 나온다기보다 같은 구절이 계속 이어집니다.
- `temperature=0.0`은 폴백을 꺼서 결과는 재현되지만(VAD 끔 값이 앞선 실행과 같음 **[확인]**), 폴백이 원래 끊어 주던 반복을 막지 못한 영향일 수 있습니다 **[추정]**.
- **small의 c07**은 원곡·보컬·VAD 끔·켬 네 조건이 모두 **같은 149단어(삽입 101)** 입니다. 입력과 무관하게 이 모델·설정 조합에서 생기는 루프로 보입니다. small이 가장 나쁘게 나온 큰 이유입니다. "큰 모델이 더 낫다"는 기대는 이 설정에서는 맞지 않았습니다 **[확인]**(원인 확정은 아님).

**관찰 3. 분리 효과는 VAD 여부에 따라 달라 보입니다.**
- **VAD 끔**: 분리가 낫다고도 나쁘다고도 말할 수 없습니다(tiny 약간 나음, base는 보컬이 훨씬 나쁨 — 대부분 루프 때문, small 비슷).
- **VAD 켬**: 보컬이 세 모델 모두 원곡보다 낫습니다. 다만 **이 비교는 공정하지 않습니다.** VAD가 원곡을 비워 버리기 때문이고, 이 차이 전부가 "분리의 인식 개선"은 아닙니다.
- 구간별로는 또렷한 곡(c01·c02)은 VAD 끔에서 원곡과 보컬이 거의 같아 분리가 필요 없었습니다.

**한계**
- 구간 8개, 정답 작성자 1명, 실행 1회(`T0.0`은 결정적), 폴백을 끈 설정. 반복 루프와 small c07은 이 설정의 영향일 수 있어 **기본 설정(폴백 켬)에서 같은지 확인해야 결론을 낼 수 있습니다.**
- 문장 단위의 오류 유형 분류는 아직 하지 않았고, 위 반복 지표는 숫자 수준의 분류입니다.

**다음 확인 (우선순위 순)**
1. **기본 설정 비교**: `TEMPS = (None,)`, `REPEATS = 3`으로 tiny·base를 돌려 루프가 폴백으로 사라지는지, 결론의 방향이 같은지 봅니다.
2. **루프 완화 옵션**: `condition_on_previous_text=False` 같은 옵션이 반복을 줄이는지 시험할 수 있습니다 **[일반 지식, 미시험]**.
3. **사용자 따라읽기 녹음 30건**에서 VAD 끔/켬 비교(`model_compare`). 말소리에서는 VAD가 원곡처럼 비우지 않는지 확인합니다.
4. 스펙트로그램(⑦)으로 원곡과 보컬의 차이를 그림으로 남깁니다.

#### 실험2) 결과 검토 (기본 설정 = 온도 폴백 켬, VAD 끔/켬, tiny·base, 3회 반복, 구간 8개·정답 231단어, 2026-11-03 밤)

> 설정: `SIZES = ["tiny","base"]`, `VADS = (False, True)`, `TEMPS = (None,)`, `REPEATS = 3` → 구간 8 × 소리 2 × 크기 2 × VAD 2 × 반복 3 = **192번 인식**. 인식 시간 합계는 약 **25.5분**(평균 8.0초)이었습니다. 실험1(평균 약 4초)보다 느린데, 폴백이 켜져 어려운 소리를 다시 인식하기 때문으로 보입니다 **[추정]**. (검토 문서·절차 문서에서는 실험1을 "설정 A", 실험2를 "설정 B"라고도 부릅니다.)

**용어 (이 칸에서 계속 쓰는 말)**

| 용어 | 뜻 |
|---|---|
| **반복 루프** (repetition loop) | 인식기가 **같은 단어·구절을 끝없이 되풀이해 적는** 현상. 방금 쓴 글을 보고 다음 단어를 고르는 방식이라, 어려운 소리에서 같은 구절이 계속 가장 그럴듯하게 뽑히면 끝까지 반복합니다 **[일반 지식]**. 정답에 없는 단어가 수십~수백 개 **삽입(I)** 으로 세어져 WER이 1을 넘습니다. 실험1에서 삽입 40개 이상인 7건은 모두 고유 단어 비율 0.01~0.04, 4단어 연속 반복 0.98~1.00이었습니다 **[확인: 지표]**. |
| **환각** (hallucination) | 소리에 없는 말을 **만들어 적는 것**. 반복 루프는 환각의 한 모양입니다. |
| **온도 폴백** (temperature fallback) | faster-whisper의 기본 동작. 인식이 나쁘다고 판단하면 **온도(무작위성)를 올려 무작위 샘플링으로 다시 인식**합니다. 반복을 끊는 데 도움이 되지만 같은 입력도 실행마다 결과가 달라질 수 있습니다. `temperature=0.0`은 폴백을 끄는 설정입니다(실험1). |
| **빈 인식** | 인식 결과가 **0단어**. 전부 삭제(D)로 세어져 WER이 정확히 1.00이 됩니다. VAD가 소리를 모두 지웠거나 모델이 "말소리 없음"으로 판단할 때 생깁니다. |
| **micro_min ~ max** | 같은 조건을 3번 돌린 micro WER의 최솟값~최댓값. **흔들림의 크기**입니다. |

**왜 이 실험을 했나**: 실험1은 폴백을 끈 **진단용 설정**이었습니다. 실제 파이프라인은 기본 설정(폴백 켬)일 가능성이 크므로, ① 실험1에서 본 현상(루프, 조건별 순위)이 기본 설정에서도 유지되는지, ② 기본 설정이 실행마다 얼마나 흔들리는지를 봤습니다.

**실행 전에 정해 둔 판단 논리와 판정** (기준은 검토 문서 13.5에 실행 전에 적어 두었습니다)

| 확인 | 실험2 결과 | 판정 |
|---|---|---|
| **① 점검**: VAD 켬 + 원곡의 빈 인식이 또 5/8인가 | tiny·base 모두 **3회 내내 같은 5구간**(c01·c03·c04·c05·c08)이 비었음 | **재현.** 원곡에는 VAD를 쓸 수 없음이 확정 **[확인]** |
| **② 루프**: 삽입 40개 이상 건수 | 실험1(tiny·base) 3/64 → 실험2 **1/96**(c05 base 원곡 1회차). micro WER도 크게 하락: base 보컬 VAD 끔 1.221 → **0.382**, tiny 원곡 VAD 끔 1.026 → **0.556** | 루프는 대부분 **폴백을 끈 탓**. 실험1은 진단용이고 파이프라인은 기본 설정으로 |
| **③ 흔들림**: 3회 폭 | 대부분 0.00~0.04. **base 원곡 VAD 끔만 0.199**(0.424~0.623): 한 구간(c05)이 회차마다 2.34 / 1.00 / 1.00, c04는 1.03 / 0.94 / 1.32 | 대체로 작지만 **어려운 구간(c04·c05)은 한 번의 값을 믿을 수 없음** |
| **④ 겹침**: [min, max]가 겹치는가 | 아래 목록 | 아래 목록 |
| **⑤ 최고 조합** | 실험2: **base + VAD 끔 + 보컬 0.382**. 실험1: base + VAD 켬 + 보컬 0.494 | **최고 조합이 바뀜.** VAD가 핵심이 아님 |

④의 비교 (겹치지 않는 차이만 인정)
- **분리 효과(VAD 끔)**: tiny는 겹침 → **구분 불가**. base는 보컬 0.368~0.398 < 원곡 0.424~0.623 → **보컬이 낮음**(0.398 대 0.424로 아슬아슬).
- **VAD 효과**: 원곡은 tiny·base 모두 켬이 **나쁨**. 보컬은 tiny 구분 불가, base는 켬이 **나쁨**(0.494~0.502 > 0.368~0.398).
- **모델 크기**: 보컬(VAD 끔)에서 base < tiny(겹치지 않음). 원곡(VAD 끔)은 겹침.
- base의 **원곡 + VAD 끔 대 보컬 + VAD 켬**은 겹침.

**세 결말 중 어디인가**: A(보컬 + VAD 켬이 최선)는 **아닙니다.** B(폴백을 켜면 루프가 사라지고 원곡 + VAD 끔이 보컬 + VAD 켬과 겹침)는 **일부만** 해당합니다. 루프가 사라진 것과 위의 겹침은 맞지만, 보컬 + VAD 끔이 base에서 원곡 + VAD 끔보다 낮아 "분리의 이득이 불확실"로만 정리할 수 없습니다. 결국 **사전에 정하지 않은 결말(A와 B 사이)** 입니다: **분리는 base에서 약하게 도움이 되고, VAD는 도움이 되지 않습니다.** 기준표가 이 경우를 예상하지 못했으므로, 아래 결정은 ④·⑤를 직접 읽어 정했습니다.

**조건별 micro WER** (실험2 3회 평균 [min ~ max], 괄호는 실험1의 `temperature=0.0` 값)

| 모델 | 소리 | VAD 끔 | VAD 켬 |
|---|---|---|---|
| tiny | 원곡 | 0.556 [0.532 ~ 0.567] (1.026) | 0.797 [0.784 ~ 0.818] (0.788) |
| tiny | 보컬 | 0.586 [0.563 ~ 0.606] (0.983) | 0.602 [0.584 ~ 0.623] (0.619) |
| base | 원곡 | 0.509 [0.424 ~ 0.623] (0.584) | 0.740 [0.740 ~ 0.740] (0.740) |
| base | 보컬 | **0.382** [0.368 ~ 0.398] (1.221) | 0.499 [0.494 ~ 0.502] (0.494) |

**관찰**
1. **폴백을 켜면 점수가 크게 좋아집니다.** VAD 끔 기준으로 base 보컬 1.221 → 0.382, tiny 원곡 1.026 → 0.556. VAD 켬은 두 설정이 거의 같은데(예: base 원곡 0.740), VAD가 디코딩 앞에서 소리를 지우기 때문입니다 **[추정]**.
2. **VAD 켬은 원곡의 가사를 지우고, 보컬에서도 일부 구간을 잘라 먹습니다.** base 보컬에서 c01 0.05 → 0.45, c08 0.00 → 0.80(3회 평균) **[확인: 구간별 WER]**. 이유는 VAD가 노래하는 소리를 말소리로 확신하지 못하는 구간이 있기 때문으로 보입니다 **[추정]**.
3. **분리는 base에서만 약한 이득**입니다: 원곡 0.509 → 보컬 0.382(약 25% 감소). tiny에서는 구분할 수 없습니다.
4. **아카펠라(c07)는 대조군답게 모든 조건에서 같습니다**: base 0.10, tiny 0.35. 반주가 없으면 분리도 VAD도 필요 없습니다 **[확인]**.
5. **VAD를 끄고도 c05는 가끔 인식이 빕니다**(tiny 원곡 3/3회, tiny 보컬 1/3회, base 원곡 2/3회). 기본 설정의 Whisper가 말소리 없음으로 판단하는 경우로 보입니다 **[추정]**.

**실험1의 결론 중 바뀐 것**

| 실험1에서 쓴 말 | 실험2 이후 |
|---|---|
| 최고 조합은 base + VAD 켬 + 보컬 | 기본 설정에서는 **base + VAD 끔 + 보컬(0.382)** |
| 큰 WER은 반복 루프(폴백을 끈 영향일 수 있음) | 폴백을 켜면 루프가 거의 사라짐 → **폴백을 끈 탓** **[확인]** |
| 분리가 VAD를 쓸 수 있게 하고 VAD가 환각을 줄인다 | **철회.** VAD는 기본 설정에서 도움이 안 됨. 분리가 base에서 약하게 **직접** 도움 |
| small이 가장 나쁨(c07 루프) | small은 이번에 안 돌려서 기본 설정에서도 그런지 **[미확인]** |

**결정 (노래 흐름 #31·#32)**
1. STT는 **기본 설정(폴백 켬), VAD 끔, 분리한 보컬, base 이상**으로 합니다. **구현 주의**: `stt.recognize_words`는 기본이 `vad_filter=True`이므로 노래에서는 `vad_filter=False`를 명시해야 합니다.
2. **노래에는 VAD 켬을 쓰지 않습니다**(원곡은 가사를 지우고, 보컬도 base에서 나빠짐).
3. `temperature=0.0`은 진단용입니다. 결과를 보고할 때는 항상 temperature 설정을 함께 적습니다.
4. **분리가 꼭 필요한가는 아직 확정이 아닙니다**(base에서만 겹치지 않고 아슬아슬). small·medium으로 확인하고 구간 수를 늘립니다.
5. 강사님의 "음성만 넘기기" 조언과 비교하면, **반주를 떼는 분리는 base에서 약한 효과가 확인됐고, 무음을 거르는 VAD는 이 데이터에서는 효과가 없거나 해로웠습니다.** VAD가 말소리용이라 노래에 맞지 않기 때문으로 보입니다 **[추정]**. 다음 면담에서 알릴 만합니다.

**한계**: 구간 8개, 정답 작성자 1명, 3회 반복(간이 기준), 어려운 구간(c04·c05)이 흔들림을 좌우함, tiny·base만 시험(small·medium은 미시험), 사용자 따라읽기 녹음의 VAD 효과는 아직 확인하지 않음. 기본 설정 자체가 무작위(폴백)이므로 값은 범위로 읽습니다.

**다음**
1. base + 보컬 + VAD 끔을 기준으로 `small`·`medium`·`large-v3-turbo`를 기본 설정으로 시험합니다(검토 문서 15.2의 순서).
2. 사용자 따라읽기 녹음 30건에서 VAD 끔/켬 비교: 말소리에서는 VAD가 가사를 지우지 않는지 봅니다.
3. ⑦ 스펙트로그램으로 원곡과 보컬의 차이를 그림으로 남깁니다.
4. 어려운 구간(c04·c05)의 오류를 문장 단위로 분류합니다.

#### 실험3) 모델 크기와 소리 × 모델 크기 (터미널 실행, 2026-11-03 밤)

> 이 시험은 노트북이 아니라 터미널(`scripts/compare_clips.py`)로 돌렸습니다. 같은 구간 8개·정답 231단어, **보컬/원곡 · VAD 끔 · 기본 설정(폴백 켬)** 입니다. 재현 명령은 맨 아래에 있습니다. 판정 기준은 시험 **전에** 검토 문서 15.2에 적어 두었습니다. (결과 파일은 `results/song_compare/size_test_*`, git 제외)

**왜 했나 (실험2에서 이어짐)**: 실험2에서 노래 STT는 "base + 분리한 보컬"이 최선이었습니다. 그러면 ① 모델을 키우면 더 좋아지는가, ② 모델이 커지면 분리가 여전히 필요한가를 확인했습니다. 첫 시험은 입력을 가장 좋았던 것(보컬)으로 **고정하고 모델만 바꿨고**, 두 번째 시험에서 가장 좋은 모델을 **원곡에도** 돌렸습니다.

**한 줄 결론**
- **모델을 키우면 좋아집니다.** 보컬 기준으로 `medium` 0.257, `small` 0.326, `base` 0.382(micro WER). 사전 기준(범위 겹침, 시간 5배 이하)으로 세 모델 모두 "낫다"입니다.
- **`medium`에서는 원곡이 보컬보다 낫고 더 빠릅니다**(0.177 대 0.257, 19.0초 대 33.7초). 반대로 `base`는 보컬이 낫습니다(원곡 0.509, 보컬 0.382). **분리가 필요한지는 모델 크기에 달려 있습니다.**
- 그래서 노래의 텍스트 추출은 **`medium` + 원곡 + VAD 끔 + 기본 설정**으로 정합니다. 분리는 STT용이 아니라 학습자용 기준 음성을 만드는 선택 기능입니다.

**1) 모델 크기 (보컬, 3회 = 1단계 1회 + 2단계 2회)**

| 모델 | micro WER 평균 [min ~ max] | 20초 구간 인식 | base 대비 시간 | 사전 기준 판정 |
|---|---|---|---|---|
| `medium` | **0.257** [0.199 ~ 0.286] | 33.7초 | 3.0배 | 낫다 |
| `small` | 0.326 [0.316 ~ 0.333] | 22.6초 | 2.0배 | 낫다 |
| `small.en` | 0.351 [0.351 ~ 0.351] | 8.4초 | 0.8배 | 낫다(아래 사후 점검에서 뒤집힘) |
| `base`(실험2의 기준선) | 0.382 [0.368 ~ 0.398] | 11.1초 | 1배 | — |

- `large-v3-turbo`는 20초 구간 인식이 57.7초로 비용 기준(base의 5배, 약 55초)을 넘어 **시험하지 않았습니다**(정확도 **[미확인]**).
- **사후 점검** (사전 기준에 없던 것, 결과를 본 뒤 추가): c05는 정답 32단어(14%)이고 모델마다 들쭉날쭉합니다(`medium` 1.00 / 0.38 / 1.00, `small.en` 0.34 ×3, base 0.78~0.84). **c05를 뺀 7구간**에서는 base 0.315 대 `medium` 0.171(낫다), `small` 0.228(낫다), `small.en` 0.352(**나쁘다**)입니다. `small.en`의 "낫다"는 c05 한 구간 덕이라 채택하지 않았습니다. `medium`의 폭 0.087은 전부 c05에서 나왔습니다.
- 오류의 모양: 회당 S / D / I는 `medium` 25.7 / 25.7 / 8.0, base 70.7 / 4.0 / 13.7. 큰 모델은 틀린 단어를 적는 대신 **건너뛰는** 쪽으로 바뀝니다 **[추정]**.

**2) 소리 × 모델 크기 (`medium` 원곡 3회)**

| 조건 | micro WER [범위] | c05 제외 7구간 | 20초 구간 인식 | S / D / I (회당) |
|---|---|---|---|---|
| **`medium` 원곡** | **0.177** [0.177 ~ 0.177] | **0.151** | **19.0초** | 26.0 / 6.0 / 9.0 |
| `medium` 보컬 | 0.257 [0.199 ~ 0.286] | 0.171 | 33.7초 | 25.7 / 25.7 / 8.0 |
| `base` 보컬 | 0.382 [0.368 ~ 0.398] | 0.315 | 11.1초 | 70.7 / 4.0 / 13.7 |
| `base` 원곡 | 0.509 [0.424 ~ 0.623] | 0.358 | 12.8초 | — |

- **`medium` 원곡은 3회가 완전히 같았고**(폭 0) 반복 루프·빈 인식이 0건입니다 **[확인]**. c05를 빼도 방향이 같습니다(0.151 < 0.171).
- 보컬 경로는 삭제가 25.7로 원곡(6.0)보다 훨씬 많습니다. 분리하면서 단어가 흐려져 큰 모델이 건너뛴 것으로 보입니다 **[추정]**(저음질 WMA를 Demucs에 넣은 영향일 수 있음, **[미확인]**).
- 구간별(3회 평균, `medium` 원곡 / 보컬): c04 0.41 / 0.59, c05 0.34 / 0.79로 어려운 구간에서 원곡이 낫고, c03은 0.12 / 0.04로 보컬이 낫고, 나머지는 거의 같습니다.
- 분리(구간당 약 45~160초) + `base`(11초)보다 `medium` 원곡(19초) 하나가 정확도와 시간 모두 낫습니다.

**결정**: 노래 STT는 `medium` + 원곡 + VAD 끔 + 기본 설정(`stt.recognize_words`는 `vad_filter=False`를 명시). `base`를 써야 할 때만 분리한 보컬. 분리는 학습자용 기준 음성(#32)을 위한 선택 기능.

**한계**: 구간 8개·곡 8개, 정답 작성자 1명이 **원곡을 듣고** 적었으므로 원곡에 유리했을 수 있습니다(분리 결과에 정답이 끌려가지 않게 하려는 설계). `medium`으로 VAD 켬을 원곡에 시험하지 않았습니다 **[미확인]**. 사후 점검은 사전 기준이 아닙니다.

**다음: 다른 곡·구간으로 검증** — 같은 결론이 처음 8구간의 우연이 아닌지 확인합니다. 계획과 판정 기준은 검토 문서 15.2에 시험 전에 적어 두었습니다.

**재현 명령** (프로젝트 루트, `PY=/opt/anaconda3/envs/voice-ai/bin/python`)
```bash
# 1단계: 모델 크기(보컬만, 각 1회) → 통과한 모델은 --repeats 2로 2단계 → 1단계와 합쳐 3회
$PY scripts/compare_clips.py --sizes small small.en medium --vad off --temperature default --repeats 1 --sources vocals --out-dir results/song_compare/size_test_stage1
# 소리 × 모델: medium을 원곡에 3회
$PY scripts/compare_clips.py --sizes medium --vad off --temperature default --repeats 3 --sources mix --out-dir results/song_compare/size_test_mix_medium
```

## ⑦ 스펙트로그램: 원곡 vs 분리한 보컬

같은 설정·같은 색 범위로 파형, 스펙트로그램, Log-Mel을 나란히 그립니다(`visualisation.plot_sound_pair`, 4강 코드). 반주가 있는 구간과 없는 구간의 에너지 분포를 비교하고, 한 줄 해석을 절차 문서 8.3절에 적습니다. 그림 제목에는 곡 이름을 넣지 않습니다.

In [ ]:
# ⑦ 원곡 vs 보컬 그림 (16 kHz 모노로 바꿔 비교; 최대 진폭을 0.9로 맞춰 밝기를 비교하기 쉽게)
SPEC_ID = "c01"
mix_p = Path(separate.WORK_DIR) / f"{SPEC_ID}_mix.wav"
voc_p = Path(separate.WORK_DIR) / f"{SPEC_ID}_vocals.wav"
if mix_p.exists() and voc_p.exists():
    def norm09(x):
        return (0.9 * x / (np.abs(x).max() + 1e-9)).astype(np.float32)
    mix16 = norm09(audio.load_audio(mix_p)[0])
    voc16 = norm09(audio.load_audio(voc_p)[0])
    fig = visualisation.plot_sound_pair([("원곡 구간(mix)", mix16), ("분리한 보컬(vocals)", voc16)],
                                        f"구간 {SPEC_ID}: 원곡 vs 분리한 보컬")
    plt.show()
else:
    print("먼저 ③ 분리를 실행하세요.")

## 기록

숫자와 관찰은 `docs/노래_구간_비교_실험_절차.md` 8절 표에 적고(가사·곡 이름 없이), 판단은 검토 문서 13.5에 옮깁니다. 이 노트북을 커밋할 때는 **출력이 없는 정리본**만 올립니다(`scripts/scan_outputs.py`로 확인).